<a href="https://colab.research.google.com/github/leeknnn/viktoriiashkorubska-41/blob/master/lab_work_20/lab_work_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import numpy as np
from scipy.optimize import linprog

# Варіант 1: Київ
# Прибуток А: 60 грн/од, Прибуток Б: 45 грн/од
# Сировина: А = 3 кг/од, Б = 300 кг/од (увага в таблиці: сировина А = 3, Б = 3, ліміт = 300)
# Перевіримо таблицю варіантів: Сировина А = 3, Сировина Б = 3, Ліміт = 300
# Час: А = 4 год/од, Б = 2 год/од, Ліміт часу = 220 год
# Електроенергія: А = 20 кВт·год/од, Б = 10 кВт·год/од, Ліміт електро = 1195 кВт·год

c = [-60, -45]  # Максимізація прибутку -> мінусуємо для linprog (мінімізація)

A_ub = [
    [3, 3],    # Сировина: 3*x1 + 3*x2 <= 300
    [4, 2],    # Час: 4*x1 + 2*x2 <= 220
    [20, 10]   # Електроенергія: 20*x1 + 10*x2 <= 1195
]

b_ub = [300, 220, 1195]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)], method="highs")

print(f"Статус оптимізації: {res.status} ({res.message})")
print(f"Оптимальні кількості (x1, x2): {res.x}")
print(f"Максимальний тижневий прибуток: {-res.fun:.2f} грн")

# Розрахунок використання ресурсів (slack)
resources_used = np.dot(A_ub, res.x)
slack = b_ub - resources_used
print(f"Використання ресурсів: {resources_used}")
print(f"Залишки ресурсів (slack): {slack}")

print("\n--- Завдання 5: What-if (Збільшення ліміту активного ресурсу на 15%) ---")
# Активний ресурс тут - Час (ліміт був 220, стає 220 * 1.15 = 253)
b_ub_active = [300, 220 * 1.15, 1195]
res_active = linprog(c, A_ub=A_ub, b_ub=b_ub_active, bounds=[(0, None), (0, None)], method="highs")
print(f"Новий прибуток після збільшення часу: {-res_active.fun:.2f} грн")
print(f"Нові обсяги випуску: {res_active.x}")

# Неактивний ресурс - Електроенергія (ліміт збільшуємо на 50% з 1195 до 1792.5)
b_ub_inactive = [300, 220, 1195 * 1.5]
res_inactive = linprog(c, A_ub=A_ub, b_ub=b_ub_inactive, bounds=[(0, None), (0, None)], method="highs")
print(f"Новий прибуток після збільшення електроенергії: {-res_inactive.fun:.2f} грн")

Статус оптимізації: 0 (Optimization terminated successfully. (HiGHS Status 7: Optimal))
Оптимальні кількості (x1, x2): [10. 90.]
Максимальний тижневий прибуток: 4650.00 грн
Використання ресурсів: [ 300.  220. 1100.]
Залишки ресурсів (slack): [ 0.  0. 95.]

--- Завдання 5: What-if (Збільшення ліміту активного ресурсу на 15%) ---
Новий прибуток після збільшення часу: 4792.50 грн
Нові обсяги випуску: [19.5 80.5]
Новий прибуток після збільшення електроенергії: 4650.00 грн


## Відповіді на контрольні питання

1. **Чому scipy.optimize.linprog завжди мінімізує і що потрібно для максимізації?**
   - Метод `linprog` за замовчуванням виконує задачу на мінімізацію. Щоб розв'язати задачу на максимізацію (як-от максимізація прибутку), коефіцієнти цільової функції обов'язково множаться на $-1$. Отримане значення цільової функції у результаті (`res.fun`) також слід помножити на $-1$, щоб отримати істинний максимальний прибуток.

2. **Чому обмеження типу «не менше» ($\ge$) потрібно домножувати на $-1$?**
   - Функція `linprog` приймає обмеження-нерівності виключно у вигляді верхньої межі ($\le$), тобто у форматі $A_{ub} x \le b_{ub}$. Якщо в задачі є умова $\ge$, домноження обох її частин на $-1$ автоматично змінює знак нерівності на протилежний без зміни математичної суті обмеження.

3. **У чому різниця між активним і неактивним обмеженням?**
   - Активне обмеження — це ресурс, який споживається повністю (залишок `slack` дорівнює нулю), утворюючи «вузьке місце» виробництва. Неактивний ресурс має позитивний запас (`slack > 0`), а отже, його поточна межа не обмежує подальше збільшення випуску продукції.

4. **Чи узгоджується зміна прибутку в Завданні 5 із приростом ресурсів?**
   - Так, повністю узгоджується. Збільшення ліміту активного ресурсу (вузького місця) призводить до зростання максимального прибутку та зміни структури випуску. Натомість розширення неактивного ресурсу не змінює ні прибутку, ні оптимальних обсягів виробництва, оскільки цей ресурс і так не використовувався повністю.